# Trabalho Prático de Mineração de Dados - UNITINS
## Clusterização com Misturas Gaussianas (Gaussian Mixture Models - GMM)
**Grupo 5:** Daniel, Vladmir, Nedson, Paulo Henrique  
**Dataset:** Rice (Cammeo and Osmancik) - UCI Machine Learning Repository (ID 545)

### Objetivos:
1. Análise Exploratória dos Dados (EDA);
2. Tratamento e normalização dos atributos numéricos;
3. Estudo do número de componentes via AIC (Akaike Information Criterion) e BIC (Bayesian Information Criterion);
4. Treinamento do modelo GMM e obtenção dos clusters;
5. Avaliação não supervisionada (Silhueta, Davies-Bouldin, Calinski-Harabasz);
6. Comparação e interpretação com as classes morfológicas originais (pureza e matriz de contingência);
7. Exportação do modelo e pipeline para inferência na aplicação web.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Carregamento dos Dados
O arquivo `Rice_Cammeo_Osmancik.arff` contém 3.810 observações de grãos de arroz das espécies Cammeo e Osmancik, caracterizados por 7 atributos morfológicos contínuos/inteiros extraídos via processamento de imagens.

In [ ]:
def load_arff(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    data_started = False
    records = []
    columns = []
    for line in lines:
        line_clean = line.strip()
        if not line_clean or line_clean.startswith('%'):
            continue
        if line_clean.lower().startswith('@attribute'):
            parts = line_clean.split()
            columns.append(parts[1])
        elif line_clean.lower().startswith('@data'):
            data_started = True
        elif data_started:
            row = [x.strip() for x in line_clean.split(',')]
            if len(row) == len(columns):
                records.append(row)
                
    df = pd.DataFrame(records, columns=columns)
    numeric_cols = columns[:-1]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

df = load_arff('Rice_Cammeo_Osmancik.arff')
print("Shape do dataset:", df.shape)
df.head()

## 2. Análise Exploratória dos Dados (EDA)
Apresentação das estatísticas descritivas básicas, verificação de valores nulos e análise de distribuição das características.

In [ ]:
print("Valores nulos por coluna:")
print(df.isnull().sum())
print("\nDistribuição das classes reais (ground truth):")
print(df['Class'].value_counts())
print("\nEstatísticas descritivas das variáveis:")
df.describe()

In [ ]:
features = ['Area', 'Perimeter', 'Major_Axis_Length', 'Minor_Axis_Length', 'Eccentricity', 'Convex_Area', 'Extent']

plt.figure(figsize=(14, 10))
for i, feat in enumerate(features, 1):
    plt.subplot(3, 3, i)
    sns.histplot(data=df, x=feat, kde=True, hue='Class', palette={'Cammeo': '#2b5c8f', 'Osmancik': '#d95f02'}, bins=30)
    plt.title(f'Distribuição de {feat}')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, cmap='Blues', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de Correlação Linear (Pearson) dos Atributos')
plt.tight_layout()
plt.show()

## 3. Pré-Processamento e Normalização
Como os dados possuem ordens de magnitude díspares (por exemplo, `Area` e `Convex_Area` na faixa de 10.000 a 18.000, e `Eccentricity` na faixa de 0.8 a 0.95), aplicamos o `StandardScaler` (média 0 e variância 1) para viabilizar o cálculo adequado da matriz de covariância das distribuições gaussianas multivariadas.

In [ ]:
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

joblib.dump(scaler, 'scaler.joblib')
print("Scaler ajustado e salvo em scaler.joblib")

## 4. Seleção de Componentes e Treinamento do Modelo GMM
Ajustamos modelos de Misturas Gaussianas variando o número de componentes de 1 a 6 e comparamos os critérios estatísticos de informação:
- **BIC (Bayesian Information Criterion)**: penaliza fortemente a complexidade do modelo (menor é melhor);
- **AIC (Akaike Information Criterion)**: mede a qualidade relativa do modelo estatístico (menor é melhor).

In [ ]:
n_components_range = range(1, 7)
bic_scores = []
aic_scores = []

for n in n_components_range:
    gmm_temp = GaussianMixture(n_components=n, covariance_type='full', random_state=42)
    gmm_temp.fit(X_scaled)
    bic_scores.append(gmm_temp.bic(X_scaled))
    aic_scores.append(gmm_temp.aic(X_scaled))

plt.figure(figsize=(9, 4))
plt.plot(n_components_range, bic_scores, marker='o', label='BIC (Bayesian Information Criterion)', color='#1f77b4')
plt.plot(n_components_range, aic_scores, marker='s', label='AIC (Akaike Information Criterion)', color='#ff7f0e')
plt.xlabel('Número de Componentes Gaussianos (K)')
plt.ylabel('Critério de Informação')
plt.title('Seleção de Hiperparâmetros: Curvas BIC e AIC')
plt.legend()
plt.tight_layout()
plt.show()

### 4.1 Treinamento do Modelo Ótimo (K=2)
Com base no problema do domínio (duas variedades de arroz) e no ponto de equilíbrio das curvas de informação, ajustamos o GMM com $K=2$ componentes com covariância completa (`full`).

In [ ]:
gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
gmm.fit(X_scaled)

clusters = gmm.predict(X_scaled)
probs = gmm.predict_proba(X_scaled)

df['Cluster'] = clusters
df['Prob_C0'] = probs[:, 0]
df['Prob_C1'] = probs[:, 1]

joblib.dump(gmm, 'gmm_model.joblib')
print("Modelo GMM treinado e salvo com sucesso em gmm_model.joblib!")

## 5. Avaliação e Métricas dos Clusters
Métricas intrínsecas (não supervisionadas) de qualidade de agrupamento:
- **Silhouette Score**: mede a coesão intra-cluster versus separação inter-cluster (-1 a 1, maior é melhor);
- **Davies-Bouldin Index**: mede a similaridade média entre clusters (menor é melhor);
- **Calinski-Harabasz Index**: razão entre dispersão entre clusters e dispersão dentro dos clusters (maior é melhor).

In [ ]:
sil = silhouette_score(X_scaled, clusters)
db = davies_bouldin_score(X_scaled, clusters)
ch = calinski_harabasz_score(X_scaled, clusters)

print("=== MÉTRICAS DE AVALIAÇÃO DE CLUSTERIZAÇÃO ===")
print(f"Silhouette Score:         {sil:.4f}")
print(f"Davies-Bouldin Index:     {db:.4f}")
print(f"Calinski-Harabasz Index:  {ch:.4f}")

### 5.1 Interpretação e Correspondência com o Domínio
Cruzamos os clusters identificados com a variedade real dos grãos para avaliar a pureza do agrupamento.

In [ ]:
cross_tab = pd.crosstab(df['Cluster'], df['Class'], margins=True)
print("Matriz de Contingência (Cluster vs Variedade Real):")
print(cross_tab)

# Mapeamento para alinhamento e cálculo de pureza
# Identifica qual variedade predomina em cada cluster
cluster_map = {}
for c in [0, 1]:
    predom = df[df['Cluster'] == c]['Class'].mode()[0]
    cluster_map[c] = predom

aligned_preds = df['Cluster'].map(cluster_map)
purity = (aligned_preds == df['Class']).mean()
print(f"\nMapeamento dos clusters: {cluster_map}")
print(f"Pureza da Clusterização: {purity*100:.2f}%")
print("\nRelatório de Classificação da Correspondência:")
print(classification_report(df['Class'], aligned_preds))

## 6. Visualização 2D via PCA (Principal Component Analysis)
Projetamos os 7 atributos em 2 componentes principais para inspecionar espacialmente a separação dos grãos e a elipse gaussiana dos clusters.

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df['PCA1'] = pca_coords[:, 0]
df['PCA2'] = pca_coords[:, 1]

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='Class', alpha=0.6, palette={'Cammeo': '#2b5c8f', 'Osmancik': '#d95f02'})
plt.title('Dados Reais Projetados em PCA (Cammeo vs Osmancik)')

plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='Cluster', palette='tab10', alpha=0.6)
plt.title('Clusters Descobertos pelo GMM em PCA')

plt.tight_layout()
plt.show()
print(f"Variância explicada acumulada pelo PCA (2 componentes): {pca.explained_variance_ratio_.sum()*100:.2f}%")